# 01 | Import & Merge Excel
## Objective
Import and integrate project, budget, expense, and milestone data from multiple Excel files into a consistent dataset for business analysis.

## Business Context
Management needs a consolidated view of project budgets, actual spending, and delivery milestones to monitor financial performance, identify delays, and support informed decision-making.

## Technical Approach
- Import Excel datasets using pandas.
- Inspect data structures and identify join keys.
- Validate data granularity and relationships.
- Aggregate financial records before merging to prevent double counting.
- Prepare an integrated dataset for subsequent analysis.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

# 01 | GitHub repository
REPOSITORY_URL = (
    "https://github.com/MayerAnnamaria/python-business-analytics-project.git"
)

# 02 | Local project directory
PROJECT_DIR = Path("/content/python-business-analytics-project")

# 03 | Clone repository if it does not exist
if not PROJECT_DIR.exists():
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(PROJECT_DIR)],
        check=True
    )

# 04 | Install project dependencies
%pip install "pandas>=2.0,<3.0" "numpy>=1.24,<3.0" "matplotlib>=3.7,<4.0" "openpyxl>=3.1,<4.0" "pytest>=7,<10"


# 05 | Set working directory
os.chdir(PROJECT_DIR)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# 06 | Configure input and output folders
INPUT_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 07 | Confirm setup
print("Project:", PROJECT_DIR)
print("Input files:", INPUT_DIR)
print("Output files:", OUTPUT_DIR)
print("Setup completed successfully!")

In [ ]:
# OPTIONAL: Uncomment to use private Excel workbooks in your Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# INPUT_DIR = Path('/content/drive/MyDrive/project-budget-performance-analytics/input')
# NOTE: You must provide all four files with compatible columns.


In [3]:
from src.business_analytics.io import read_sources
from src.business_analytics.cleaning import clean_sources
from src.business_analytics.budget import budget_analysis
raw = read_sources(INPUT_DIR)
for name, frame in raw.items():
    print(f'{name}: {len(frame)} rows; columns = {list(frame.columns)}')
    display(frame.head(3))


projects: 25 rows; columns = ['project_id', 'project_name', 'department', 'project_manager', 'status', 'start_date', 'planned_end_date', 'actual_end_date', 'priority']


,project_id,project_name,department,project_manager,status,start_date,planned_end_date,actual_end_date,priority
0,PRJ-001,ERP integration – IT,IT,Alex Morgan,Completed,2025-01-14,2025-04-21,2025-04-09,Medium
1,PRJ-002,Cloud migration – Operations,Operations,Jordan Lee,Completed,2025-01-27,2025-05-21,2025-05-21,Medium
2,PRJ-003,CRM rollout – Finance,Finance,Taylor Singh,Completed,2025-02-09,2025-06-20,2025-06-25,Medium


budget_plan: 96 rows; columns = ['project_id', 'cost_category', 'planned_budget']


,project_id,cost_category,planned_budget
0,PRJ-001,Labor,26700
1,PRJ-001,Software,12800
2,PRJ-001,External Services,8000


actual_costs: 289 rows; columns = ['transaction_id', 'project_id', 'cost_category', 'transaction_date', 'actual_cost', 'description']


,transaction_id,project_id,cost_category,transaction_date,actual_cost,description
0,TXN-001-1-1,PRJ-001,Labor,2025-02-24,8010.0,Invoice batch 1
1,TXN-001-1-2,PRJ-001,Labor,2025-03-17,8010.0,Invoice batch 2
2,TXN-001-1-3,PRJ-001,Labor,2025-04-07,8010.0,Invoice batch 3


milestones: 96 rows; columns = ['milestone_id', 'project_id', 'milestone_name', 'planned_date', 'actual_date', 'status']


,milestone_id,project_id,milestone_name,planned_date,actual_date,status
0,MS-001-1,PRJ-001,Kickoff,2025-02-02,2025-02-09,Completed
1,MS-001-2,PRJ-001,Requirements signed off,2025-02-22,2025-03-10,Completed
2,MS-001-3,PRJ-001,Build & testing,2025-03-13,2025-03-16,Completed


**Grain matters:** Project master has one row per project, plans one per project/category, expenses one per transaction, milestones one per milestone. Joining raw line items to each other can multiply spend and budgets. Aggregate first.


In [6]:
# =============================================================================
# 03 | BUDGET DATA INTEGRATION & ANALYSIS
# =============================================================================
# Clean the source datasets and calculate budget performance.
#
# The budget_analysis() function:
#   1. Aggregates planned budgets and actual expenses by project and category.
#   2. Merges the aggregated financial datasets.
#   3. Calculates budget variances and performance indicators.
#   4. Merges project-level results with project master data.
#
# Outputs:
#   category   - Budget performance by project and cost category.
#   by_project - Budget performance by project.

# Notes:
# Clean the source datasets, calculate budget performance by cost category and project, and
# Grain (also called data granularity) means what one row represents in a dataset.
# Data granularity: Each source has a different row-level structure.
# Aggregate budgets and expenses to the same project-level grain
# =============================================================================

# Clean source data
tables = clean_sources(raw)

# Aggregate, merge, and analyze financial data
category, by_project = budget_analysis(tables)

# Preview the results
display(category.head(10))
display(by_project.head(10))


,project_id,cost_category,planned_budget,actual_cost,variance,variance_pct,budget_status
85,PRJ-022,Labor,62400,74880.00,12480.00,20.0,Over budget
41,PRJ-011,Labor,43700,53532.51,9832.51,22.5,Over budget
61,PRJ-016,Labor,52200,61335.00,9135.00,17.5,Over budget
81,PRJ-021,Labor,60700,68287.50,7587.50,12.5,Over budget
17,PRJ-005,Labor,33500,40200.00,6700.00,20.0,Over budget
78,PRJ-020,Software,28300,34667.49,6367.49,22.5,Over budget
37,PRJ-010,Labor,42000,48300.00,6300.00,15.0,Over budget
57,PRJ-015,Labor,50500,55550.01,5050.01,10.0,Over budget
54,PRJ-014,Software,23400,28080.00,4680.00,20.0,Over budget
74,PRJ-019,Software,27500,31625.01,4125.01,15.0,Over budget


,project_id,planned_budget,actual_cost,variance,variance_pct,project_name,department,status,budget_status
21,PRJ-022,118600,130980.00,12380.00,10.4,Workflow redesign – Customer Service,Customer Service,On Hold,Over budget
10,PRJ-011,83000,93752.52,10752.52,13.0,CRM rollout – Marketing,Marketing,Completed,Over budget
19,PRJ-020,112100,122040.00,9940.00,8.9,Reporting automation – Operations,Operations,In Progress,Over budget
15,PRJ-016,99300,107177.52,7877.52,7.9,Vendor onboarding – Customer Service,Customer Service,In Progress,Over budget
4,PRJ-005,63600,70245.00,6645.00,10.4,Service portal – Marketing,Marketing,Completed,Over budget
20,PRJ-021,115300,121802.49,6502.49,5.6,Service portal – Finance,Finance,In Progress,Over budget
13,PRJ-014,92700,98607.48,5907.48,6.4,Workflow redesign – Operations,Operations,Completed,Over budget
2,PRJ-003,57100,62160.00,5060.00,8.9,CRM rollout – Finance,Finance,Completed,Over budget
9,PRJ-010,79800,84150.00,4350.00,5.5,Cloud migration – Customer Service,Customer Service,Completed,Over budget
3,PRJ-004,60400,63792.51,3392.51,5.6,Reporting automation – Customer Service,Customer Service,Completed,Over budget
